# IPL Cricket Fielding Performance Analysis

This notebook evaluates the fielding contributions of three selected players from T20 match data using a custom performance score. The goal is to quantify fielding impact in a way that combines clean picks, accurate throws, catches, run-outs, direct hits, and run-saving contributions.

The workbook loads a CSV dataset, filters for the chosen players, calculates the required metrics, and then creates a summary table and charts for a short walkthrough.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

data_path = "sample_fielding_data.csv"
fielding_df = pd.read_csv(data_path)

# Ensure the key columns are imported with the correct data types
fielding_df["Player Name"] = fielding_df["Player Name"].astype(str)
fielding_df["Team"] = fielding_df["Team"].astype(str)
fielding_df["Pick"] = fielding_df["Pick"].astype(str).str.upper()
fielding_df["Throw"] = fielding_df["Throw"].astype(str).str.upper()
fielding_df["Runs"] = pd.to_numeric(fielding_df["Runs"], errors="coerce").fillna(0)
fielding_df["Ball Count"] = pd.to_numeric(fielding_df["Ball Count"], errors="coerce").fillna(0).astype(int)
fielding_df["Position"] = pd.to_numeric(fielding_df["Position"], errors="coerce").fillna(0).astype(int)
fielding_df["Overcount"] = pd.to_numeric(fielding_df["Overcount"], errors="coerce").fillna(0).astype(int)
fielding_df["Venue"] = fielding_df["Venue"].astype(str)
fielding_df["Stadium"] = fielding_df["Stadium"].astype(str)

# Convert action flags into numeric 1/0 values
for col in ["Catch_Successful", "Dropped_Catch", "Stumping", "Run_Out", "Missed_Run_Out", "Direct_Hit"]:
    fielding_df[col] = fielding_df[col].astype(str).str.upper()
    fielding_df[col] = fielding_df[col].map({"Y": 1, "YES": 1, "TRUE": 1, "1": 1, "N": 0, "NO": 0, "FALSE": 0, "0": 0}).fillna(0).astype(int)

selected_players = ["Rohit Sharma", "Suryakumar Yadav", "Hardik Pandya"]
analysis_df = fielding_df[fielding_df["Player Name"].isin(selected_players)].copy()
analysis_df.head()

## 1) Data structure and player filter

The CSV contains the event-level fielding actions. Each row describes one ball or fielding action. The notebook then filters to the selected three players and prepares the metrics that will feed the scoring model.

In [ ]:
# Aggregate per player
player_summary = (
    analysis_df.groupby("Player Name", as_index=False)
    .agg(
        Position=("Position", "first"),
        CP=("Pick", lambda s: (s == "Y").sum()),
        GT=("Throw", lambda s: (s == "Y").sum()),
        C=("Catch_Successful", "sum"),
        DC=("Dropped_Catch", "sum"),
        ST=("Stumping", "sum"),
        RO=("Run_Out", "sum"),
        MRO=("Missed_Run_Out", "sum"),
        DH=("Direct_Hit", "sum"),
        RS=("Runs", "sum")
    )
)

# Compute performance score using the required formula
player_summary["PS"] = (
    player_summary["CP"] * 1
    + player_summary["GT"] * 1
    + player_summary["C"] * 3
    - player_summary["DC"] * 3
    + player_summary["ST"] * 3
    + player_summary["RO"] * 3
    - player_summary["MRO"] * 2
    + player_summary["DH"] * 2
    + player_summary["RS"]
)

# Reorder columns for presentation
player_summary = player_summary[[
    "Player Name", "Position", "CP", "GT", "C", "DC", "ST", "RO", "MRO", "DH", "RS", "PS"
]]

player_summary

## 2) Performance score formula

The notebook uses the exact score formula requested in the task:

$PS = (CP 	imes 1) + (GT 	imes 1) + (C 	imes 3) - (DC 	imes 3) + (ST 	imes 3) + (RO 	imes 3) - (MRO 	imes 2) + (DH 	imes 2) + RS$

where each fielding metric is aggregated per player and then combined into a single comparison score.

In [ ]:
# Display the full summary table
summary_table = player_summary.copy()
summary_table = summary_table.sort_values("PS", ascending=False).reset_index(drop=True)
summary_table

In [ ]:
# Bar chart: Performance Score comparison
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=summary_table, x="Player Name", y="PS", hue="Player Name", dodge=False, palette="viridis", legend=False)
ax.set_title("Performance Score Comparison by Player")
ax.set_xlabel("Player")
ax.set_ylabel("Performance Score (PS)")
for container in ax.containers:
    ax.bar_label(container, padding=3, fmt="%.0f")
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar: action-type breakdown by player
stack_df = summary_table[[
    "Player Name", "CP", "GT", "C", "DC", "ST", "RO", "MRO", "DH"
]].melt(id_vars="Player Name", var_name="Action Type", value_name="Value")

plt.figure(figsize=(11, 6))
ax = sns.barplot(data=stack_df, x="Player Name", y="Value", hue="Action Type", palette="Set2")
ax.set_title("Fielding Action Breakdown by Player")
ax.set_xlabel("Player")
ax.set_ylabel("Count")
plt.legend(title="Action Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3) Summary insights

The final section converts the numbers into a short business-style summary: the best overall fielder, the main strength of each player, and the general impact of saved runs.

In [ ]:
best_player = summary_table.iloc[0]
best_metric = summary_table.drop(columns=["Player Name", "Position", "RS", "PS"]).max().idxmax()

print("Top fielder:", best_player["Player Name"])
print("Top fielder score:", best_player["PS"])
print("")

for _, row in summary_table.iterrows():
    strongest_metric = row[["CP", "GT", "C", "ST", "RO", "DH"]].idxmax()
    if row["RS"] >= 0:
        run_message = f"saved +{row['RS']} runs with smart fielding support."
    else:
        run_message = f"conceded {row['RS']} runs in defensive fielding phases."
    print(f"{row['Player Name']} strongest area: {strongest_metric}; {run_message}")

print("")
print("Key insight:")
print(
    f"{best_player['Player Name']} leads the fielding chart because they combine high positional control, effective throws, and run-saving actions."
    "The aggregate table also highlights how direct hits, run-outs, and clean picks improve a player’s overall fielding value beyond raw run totals."
)